# 20261111 · 课程串讲：检索质量进阶 —— 混合检索 · 融合排序 · 精排 · 改写 · 评估

上一讲（1109）把 RAG 的七步流水线跑通了：加载 → 清洗 → 切块 → 嵌入 → Chroma → 检索 → 生成。
这一讲不动其余环节，只升级其中最弱的一环——**检索**。语料仍是《峨眉山讲解词.docx》，才能把效果差异归因到检索方法本身。

| 环节 | 1109 的做法 | 1111 的升级 | 课堂文件 |
|---|---|---|---|
| 向量库 | 嵌入式 Chroma | 服务化 Milvus（gRPC 19530） | `test1.py` `test3.py` |
| 稀疏检索 | 无 | BM25 + jieba 中文分词 | `test4.py` |
| 索引结构 | 单个稠密向量字段 | dense + sparse 双字段，服务端内置 BM25 | `test5.py` |
| 多路融合 | 无 | weighted 加权 / RRF 倒数排名融合 | `test6.py` |
| 精排 | 无 | Cross-Encoder 对 top-n 重新打分 | `test6.py` |
| 查询 | 原样检索 | MultiQuery 一个问题改写成多个 | `test7.py` |
| 交付形态 | 检索器直接调用 | 检索器转工具，交给 Agent 路由 | `test2.py` |
| 效果评估 | 人工看答案 | RAGAS：Faithfulness / ContextRecall | `ragas_test.py` |

## 执行说明

- 环境：`week23` 的 Python 3.11 虚拟环境（`langchain-learning (3.11.9)`），依赖含 `langchain-milvus`、`pymilvus`、`jieba`、`rank-bm25`、`ragas`。
- 云端配置沿用 `week23/.env`（`LLM_PROVIDER=qwen`）：对话 `qwen3.8-flash`，嵌入 `text-embedding-v4`（1024 维，每批最多 10 条）。
- 向量库用本地 Milvus：`docker compose -f milvus/docker-compose.yml up -d`，gRPC 端口 19530。课堂里的 `192.168.2.197` 只是教室服务器地址，本地一律映射成 `127.0.0.1`。
- 三个开关默认关闭，离线也能读：`RUN_LIVE`（云端嵌入/对话）、`RUN_MILVUS`（向量库读写）、`RUN_EVAL`（RAGAS 裁判）。
- 离线可执行：语料加载与切块、jieba 分词、BM25 检索、手写 RRF 融合。
- 下文保留的输出来自一次把三个开关全部打开的实际运行；交付时开关保持 `False`，避免误跑产生费用。


## 0. 环境自检

先确认「资料在哪、依赖齐不齐、云端和 Milvus 通不通」。本讲涉及三个服务：

| 服务 | 地址 | 用途 |
|---|---|---|
| 千问兼容接口 | `week23/.env` 的 `QWEN_BASE_URL` | 嵌入、生成、MultiQuery 改写、RAGAS 裁判 |
| 本地 Milvus | `http://127.0.0.1:19530` | 稠密 + 稀疏双字段检索 |
| DashScope rerank | `https://dashscope.aliyuncs.com` | 云端 Cross-Encoder 精排，替代课堂本地 `rerank_qwen` |

课堂原本由 Ollama（`192.168.1.105:11434`）提供 `qwen3-embedding:0.6b` 与 `qwen3.5:9b`，本讲全部换成云端配置。


In [1]:
import importlib.metadata as md
import math
import sys
import time
import warnings
from pathlib import Path

# ---- 三个开关（默认全关，离线也能读完整本） ----
RUN_LIVE = False    # 云端嵌入 + 对话：week23/.env 的千问配置，按 token 计费
RUN_MILVUS = False  # 本地 Milvus 读写：先 docker compose -f milvus/docker-compose.yml up -d
RUN_EVAL = False    # RAGAS 评测：每个样本 2 次裁判调用，最贵

warnings.filterwarnings("ignore", message="`langchain-community` is being sunset")
warnings.filterwarnings("ignore", message="IProgress not found")

MILVUS_URI = "http://127.0.0.1:19530"   # 课堂是 http://192.168.2.197:19530
EMBED_MODEL = "text-embedding-v4"
EMBED_DIM = 1024
DATA_ROOT = None    # 从别的目录启动时，填 week25 的绝对路径

start_dir = Path(DATA_ROOT).expanduser().resolve() if DATA_ROOT else Path.cwd()
docx_candidates = [
    start_dir / "20261109RAG" / "峨眉山讲解词.docx",
    start_dir / "峨眉山讲解词.docx",
]
DOCX = next((path for path in docx_candidates if path.is_file()), None)
if DOCX is None:
    raise FileNotFoundError("找不到《峨眉山讲解词.docx》。请把 DATA_ROOT 指向 week25 或 20261109RAG。")

NOTEBOOK_DIR = DOCX.parent.parent                    # week25
LAB_CONFIG_DIR = NOTEBOOK_DIR.parent / "week23" / "20261027"

print("== 包版本 ==")
for name in ("langchain", "langchain-core", "langchain-community", "langchain-milvus",
             "langchain-classic", "langchain-openai", "pymilvus", "jieba", "rank-bm25",
             "ragas", "sentence-transformers"):
    try:
        print(f"  {name:24s}{md.version(name)}")
    except md.PackageNotFoundError:
        print(f"  {name:24s}[未安装]")
print(f"资料：{DOCX.name}（{DOCX.stat().st_size / 1024:.0f} KB）")
print(f"本地配置目录：{LAB_CONFIG_DIR}")

embedding_model = llm = None


== 包版本 ==
  langchain               1.4.7
  langchain-core          1.7.0
  langchain-community     0.4.2
  langchain-milvus        0.4.0
  langchain-classic       1.0.8
  langchain-openai        1.7.3
  pymilvus                3.0.3
  jieba                   0.42.1
  rank-bm25               0.2.2
  ragas                   0.4.3
  sentence-transformers   [未安装]
资料：峨眉山讲解词.docx（156 KB）
本地配置目录：D:\project\step1\week23\20261027


In [2]:
if RUN_LIVE:
    if str(LAB_CONFIG_DIR) not in sys.path:
        sys.path.insert(0, str(LAB_CONFIG_DIR))
    import lab_config
    from langchain_openai import OpenAIEmbeddings

    if lab_config.PROVIDER != "qwen":
        raise ValueError("本讲按 week23/.env 的 LLM_PROVIDER=qwen 编写，请先切换配置。")

    llm = lab_config.get_model(temperature=0)
    embedding_model = OpenAIEmbeddings(
        model=EMBED_MODEL,
        base_url=lab_config.BASE_URL,
        api_key=lab_config.API_KEY,
        check_embedding_ctx_length=False,
        dimensions=EMBED_DIM,
        chunk_size=10,
    )
    print(f"对话模型：{lab_config.MODEL}")
    print(f"嵌入模型：{EMBED_MODEL} / {EMBED_DIM} 维（每批 10 条）")
else:
    print("离线模式：不加载云端模型；第 3、5~10 节的联网单元格会打印跳过说明。")


对话模型：qwen3.8-flash
嵌入模型：text-embedding-v4 / 1024 维（每批 10 条）


In [3]:
if RUN_LIVE:
    probe = embedding_model.embed_query("峨眉山万佛顶海拔3099米")
    if len(probe) != EMBED_DIM or not all(math.isfinite(x) for x in probe):
        raise ValueError("嵌入接口返回维度错误或非有限数值。")
    print(f"嵌入探测通过：{len(probe)} 维")
    print("对话探测：", llm.invoke("请只回复 OK。").content.strip())

if RUN_MILVUS:
    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)
    print(f"Milvus {client.get_server_version()}；现有集合：{client.list_collections()}")
    client.close()
else:
    print("Milvus 检查跳过：docker compose -f milvus/docker-compose.yml up -d 后把 RUN_MILVUS 设为 True")


嵌入探测通过：1024 维


对话探测： OK


Milvus 3.0-20261030-af15c80751；现有集合：['demo_rag']


## 1. 本讲地图：四层新增能力

```mermaid
flowchart TB
    Q[用户问题] --> MQ[第 8 节 MultiQuery 改写]
    MQ --> D[稠密检索<br/>text-embedding-v4]
    MQ --> S[稀疏检索<br/>BM25 / 内置 chinese analyzer]
    D --> F[第 6 节 融合：weighted / RRF]
    S --> F
    F --> RR[第 7 节 Cross-Encoder 精排 top-n]
    RR --> P[拼 Prompt → 生成答案]
    P --> E[第 10 节 RAGAS 评估]
```

| 节 | 解决什么问题 | 课堂文件 |
|---|---|---|
| 3 | 换服务化向量库，支持稀疏 + 稠密共存 | `test1.py` `test3.py` |
| 4 | 关键词、数字、专名的精确检索 | `test4.py` |
| 5 | 一次写入，两路索引 | `test5.py` |
| 6 | 两路结果怎么合并排序 | `test6.py` |
| 7 | 合并后的候选怎么精排 | `test6.py` |
| 8 | 用户问法不好时，先改写再检索 | `test7.py` |
| 9 | 让 Agent 决定何时检索 | `test2.py` |
| 10 | 用两个指标把「检索差」和「生成差」分开 | `ragas_test.py` |


### 为什么单路稠密检索不够用

稠密向量擅长「意思相近」，不擅长「字面精确」：

| 查询 | 稠密向量的表现 | BM25 的表现 |
|---|---|---|
| 「海拔 3099 米的是哪座峰」 | 能召回「山峰 / 海拔」语义相关段落，但 3099 不是语义单元，容易漂到 3077 米的金顶 | 数字与「万佛顶」字面共现的段落，直接命中 |
| 「圣积晚钟」这类专名 | 专名被压成语义模糊的向量，容易召回其它钟、其它景 | 整词命中且 IDF 高（全库少见），权重极大 |
| 「哪里最高」这种口语问法 | 同义改写（最高 / 极巅 / 之巅）都能匹配 | 只认「最高」这个词，改写后就漏 |

所以工业界的常规做法是 **两路召回 + 融合 + 精排**：稠密兜住语义，稀疏兜住字面，重排模型再精挑。


## 2. 语料与切块：同一份资料，才能归因

换检索栈、不换语料。切块参数与课堂一致：`chunk_size=500`、`chunk_overlap=70`、中文标点优先。

中文句末标点必须显式写进 `separators`——默认分隔符是英文的 `\n\n / \n / " "`，对中文长段落几乎切不开。


In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

t0 = time.perf_counter()
raw_docs = Docx2txtLoader(str(DOCX)).load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=70,
    separators=["\n\n", "\n", "。", "！", "？", " ", ""],
)
chunks = text_splitter.split_documents(raw_docs)

for i, doc in enumerate(chunks, start=1):
    doc.metadata.update({"source": DOCX.name, "chunk_id": i})

lengths = [len(d.page_content) for d in chunks]
raw_chars = sum(len(d.page_content) for d in raw_docs)
print(f"原文 {len(raw_docs)} 篇 / {raw_chars} 字")
print(f"切块 {len(chunks)} 块，平均 {sum(lengths) / len(lengths):.0f} 字，最长 {max(lengths)} 字，"
      f"用时 {time.perf_counter() - t0:.2f}s")
print("\n== 第 1 块 ==")
print(chunks[0].page_content[:160])
print("\n== 中间一块 ==")
print(chunks[len(chunks) // 2].page_content[:160])


原文 1 篇 / 73900 字
切块 201 块，平均 377 字，最长 498 字，用时 0.04s

== 第 1 块 ==
峨眉山导游词

峨眉山概况

峨眉山，雄踞四川盆地西南边缘，素有“仙山佛国”、“植物王国”、“动物乐园”和“地质博物馆”、“全国武术之乡”之美誉。其核心景区面积达154平方公里，最高峰万佛顶海拔3099米，主峰金顶3079米，以 “秀甲天下”的山水风光，丰富的动植物资源，厚重的佛教文化和独具特色的人文景观享誉中外，被联

== 中间一块 ==
（十四）万年寺：无梁砖殿，第一山碑

万年寺，海拔1020米，踞观心岭下，门迎玉女、大坪、牛心、钵孟诸峰，是峨眉山最早的六大古寺之一。

东晋时期，由于峨眉山开始信仰佛教、崇拜普贤，晋隆安三年（公元399年），由道安门人慧远之弟慧持大师以“欲观瞻峨眉，振锡岷岫”，辞远入蜀，得到了刺史毛璩的帮助，不辞辛苦地来到峨眉山，第


`chunk_size=500` 配 `chunk_overlap=70` 是课堂上反复用的经验值：重叠约 14%，落在「20%~30%」的下沿，适合讲解词这种段落间有承接关系的文本。

- 块太大：检索命中后塞进 Prompt 的信息里废话多，Faithfulness 很容易被噪声拖下去；
- 块太小：答案所需的两句被切成两块，单块证据不全，ContextRecall 吃亏；
- 块是检索的最小单位，`k` 个块就是模型能看到的上限。

下一讲若要做 Parent-Child（小块检索、大块喂给模型），就需要保留这一节的 `chunk_id` 来做父子映射。


## 3. Milvus：把向量库做成服务（`test1.py` / `test3.py`）

课堂第一步先测连接，因为后面所有操作都走网络：

```python
client = MilvusClient(uri="http://192.168.2.197:19530")
print("Milvus 版本：", client.get_server_version())
```

| 维度 | Chroma（1109） | Milvus（1111） |
|---|---|---|
| 部署形态 | 进程内嵌入式，`persist_directory` 落盘 | 独立服务，gRPC 19530 / 指标 9091 |
| 多语言 | Python 为主 | Python / Java / Go / Node… |
| 索引 | HNSW / IVF 等 | 更丰富，`AUTOINDEX` 自动选型 |
| 标量过滤 | where 语法 | 表达式 + 标量索引 |
| 稀疏向量 | 不支持 | **原生 sparse 字段**，可挂 BM25 内置函数 |
| 一致性 | 本地直读 | `Strong` / `Session` / `Bounded` 可选 |

一致性级别（`test5.py` 注释原文的整理）：

| 级别 | 语义 | 适用 |
|---|---|---|
| `Strong` | 写入后立刻对所有人可见（最严格、最慢） | 对账、强一致业务 |
| `Session` | 同一会话内立刻读到自己的写入 | 实验、写完就查 |
| `Bounded` | 有一定延迟上界，折中 | 大部分在线业务 |


In [5]:
if RUN_MILVUS:
    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)
    print("Milvus 版本：", client.get_server_version())
    print("已有集合：", client.list_collections())
    client.close()
else:
    print("跳过：需要 RUN_MILVUS=True（先启动 milvus/docker-compose.yml）")


Milvus 版本： 3.0-20261030-af15c80751
已有集合： ['demo_rag']


In [6]:
from langchain_milvus import Milvus

DEMO_TEXTS = [
    "万佛顶海拔3099米，是峨眉山最高处。",
    "四面十方普贤像通高48米，总重660吨。",
    "金顶祥光由日出、云海、佛光、圣灯组成。",
    "九老洞全程长1505米，能直立行走的仅120米。",
    "峨眉山猴群约15群、500余只，属藏酋猴。",
]

demo_db = None
if RUN_LIVE and RUN_MILVUS:
    t0 = time.perf_counter()
    demo_db = Milvus.from_texts(
        texts=DEMO_TEXTS,
        embedding=embedding_model,
        connection_args={"uri": MILVUS_URI},
        collection_name="emei_demo",
        drop_old=True,
    )
    print(f"建集合 emei_demo，写入 {len(DEMO_TEXTS)} 条，用时 {time.perf_counter() - t0:.1f}s")

    for i, doc in enumerate(demo_db.similarity_search("哪座山峰最高，海拔多少", k=2), start=1):
        print(f"[{i}] {doc.page_content}")
else:
    print("跳过：需要 RUN_LIVE=True 且 RUN_MILVUS=True")


建集合 emei_demo，写入 5 条，用时 1.8s
[1] 万佛顶海拔3099米，是峨眉山最高处。
[2] 峨眉山猴群约15群、500余只，属藏酋猴。


### 三个 API 家族与一个版本坑

| 用法 | 说明 |
|---|---|
| `Milvus.from_texts(texts, embedding, ...)` | 纯文本快速建库（课堂 `test3.py`），自动补 id |
| `Milvus.from_documents(docs, embedding, ...)` | 保留 `metadata`，能做标量过滤和溯源 |
| `db.add_documents(docs)` | 往已有集合追加（配合 `drop_old=False`） |
| `db.similarity_search(q, k)` | 直接拿结果 |
| `db.as_retriever(search_kwargs=...)` | 交给链或 Agent 用 |

Milvus 3.0 的行为差异：用原生 `pymilvus` 的 `client.insert(...)` 之后，需要 `client.flush(collection)` 才能稳定检索到刚写入的数据；而 `langchain_milvus.from_documents` 封装后的写入实测可以直接查到，所以本讲不额外 flush。


## 4. BM25：字面匹配的数学（`test4.py`）

BM25 是 TF-IDF 的改进版，给每个 query 词打分后求和：

$$
\text{score}(q,d)=\sum_{t\in q}\text{IDF}(t)\cdot
\frac{f(t,d)\,(k_1+1)}{f(t,d)+k_1\left(1-b+b\cdot\frac{|d|}{\mathrm{avgdl}}\right)},
\qquad
\text{IDF}(t)=\ln\!\left(1+\frac{N-n_t+0.5}{n_t+0.5}\right)
$$

| 符号 | 含义 | 直觉 |
|---|---|---|
| $f(t,d)$ | 词 $t$ 在文档 $d$ 中的词频 | 出现越多分越高，但被 $k_1$ 压住，避免「复读文档」刷分 |
| $|d|$ / $\mathrm{avgdl}$ | 文档长度 / 全库平均长度 | 长文档天然多命中，用 $b$ 做惩罚 |
| $k_1$ | 词频饱和系数，常取 1.2~2.0 | 词频从 1 到 10 的增益逐渐变平 |
| $b$ | 长度归一化强度，常取 0.75 | $b=0$ 不考虑长度，$b=1$ 全额惩罚 |
| $N$ / $n_t$ | 文档总数 / 含词 $t$ 的文档数 | 越少见的词 IDF 越大——这正是专名、型号、数字的优势 |

中文必须先分词：整句「车顶行李架载重」当成一个 term 会 OOV，分词后「行李架」才能和文档里的「行李架」对上。


In [7]:
import logging

import jieba

jieba.setLogLevel(logging.WARNING)   # 关掉 jieba 的加载日志


def chinese_tokenize(text: str) -> list[str]:
    """中文分词：BM25 的 term 就是分词结果（课堂 test4.py 的写法）。"""
    return [word for word in jieba.cut(text) if word.strip()]


question = "万佛顶的海拔是多少米"
print("整句不分词：", [question])
print("jieba 分词后：", chinese_tokenize(question))

from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks,
    preprocess_func=chinese_tokenize,
)
bm25_retriever.k = 3

print(f"\n问题：{question}")
for rank, doc in enumerate(bm25_retriever.invoke(question), start=1):
    print(f"[{rank}] chunk#{doc.metadata['chunk_id']}  {doc.page_content[:70]}...")


整句不分词： ['万佛顶的海拔是多少米']


jieba 分词后： ['万佛', '顶', '的', '海拔', '是', '多少', '米']



问题：万佛顶的海拔是多少米
[1] chunk#171  （三十一）千佛顶万佛顶：大千世界，万佛朝宗

万佛顶，海拔3099米，巍峨屹立在“大光明山”之巅。

现在我们所处的位置是千佛顶，因顶上原有...
[2] chunk#195  万佛朝宗

万佛顶，巍峨屹立在“大光明山”之巅。这里是中国佛教四大名山以及风景名胜区海拔最高的自然生态保护区。大家眼前的万佛阁，通高21米，...
[3] chunk#161  四面十方普贤像可以称得上佛教造像史上的奇迹，完美地体现了“千座名山一座顶”的佛教修行理念，是峨眉山普贤道场“功德圆满、行愿合一”的象征。

...


读 BM25 的结果时要抓两点：

1. **命中理由可解释**：命中的块一定和 query 有共同 term。合不合理，看分词就能判断；这也是为什么中文场景要维护自定义词典（寺院名、景点名、型号）。
2. **只认字面**：问「哪里最高」时 BM25 不知道「最高」= 「极巅」，向量模型知道。两路各有一半正确，这正是第 5、6 节要做混合检索的理由。

BM25 还有两个工程优点：不需要 GPU、不需要网络，`rank_bm25` 纯 Python 就能跑；所以它常被当作「免费的第二路召回」。


## 5. 混合检索：一个 collection，两路向量（`test5.py`）

```mermaid
flowchart LR
    T[同一段文本] --> E[embedding 模型<br/>dense 字段 1024 维]
    T --> A[服务端 BM25 analyzer<br/>sparse 字段：term → 权重]
    E --> C[(Collection<br/>dense + sparse)]
    A --> C
    C --> H[hybrid_search<br/>一次查询两路 + ranker 融合]
```

| 参数 | 含义 | 课堂取值 |
|---|---|---|
| `builtin_function=BM25BuiltInFunction(analyzer_params={"type": "chinese"})` | 服务端自动把文本字段转成稀疏向量，不用自己算 BM25 | `{"type": "chinese"}` |
| `vector_field=["dense", "sparse"]` | collection 里两个向量字段，顺序即稠密在前 | 与 `embedding`、`builtin_function` 一一对应 |
| `consistency_level` | 写入可见性 | `"Session"` |
| `drop_old` | `True` 重建集合，`False` 追加 | 课堂 `False`；本讲为了可重复运行用 `True` |

一个关键设计：**稠密向量由客户端（`text-embedding-v4`）算，稀疏向量由 Milvus 服务端算**。
所以建库时既要传 `embedding_function`，又要在 `builtin_function` 里声明 BM25，两路才能同时写入。


In [8]:
from langchain_milvus import BM25BuiltInFunction, Milvus

DENSE_COLLECTION = "emei_dense"
HYBRID_COLLECTION = "emei_hybrid"
dense_db = hybrid_db = None

if RUN_LIVE and RUN_MILVUS:
    t0 = time.perf_counter()
    dense_db = Milvus.from_documents(
        documents=chunks,
        embedding=embedding_model,
        connection_args={"uri": MILVUS_URI},
        collection_name=DENSE_COLLECTION,
        drop_old=True,
    )
    print(f"稠密库 {DENSE_COLLECTION}：{len(chunks)} 块，{time.perf_counter() - t0:.1f}s")

    t0 = time.perf_counter()
    hybrid_db = Milvus.from_documents(
        documents=chunks,
        embedding=embedding_model,
        connection_args={"uri": MILVUS_URI},
        builtin_function=BM25BuiltInFunction(analyzer_params={"type": "chinese"}),
        vector_field=["dense", "sparse"],
        collection_name=HYBRID_COLLECTION,
        consistency_level="Session",
        drop_old=True,
    )
    print(f"混合库 {HYBRID_COLLECTION}：{len(chunks)} 块（dense + sparse），{time.perf_counter() - t0:.1f}s")
else:
    print("跳过建库：需要 RUN_LIVE=True 且 RUN_MILVUS=True")


稠密库 emei_dense：201 块，8.6s


混合库 emei_hybrid：201 块（dense + sparse），8.4s


In [9]:
QUERIES = [
    "3099米海拔的是哪座峰",   # 数字 + 专名，BM25 占优
    "峨眉山哪座山峰最高",       # 语义提问，稠密占优
]


def show(title, docs, limit=3):
    print(f"  {title}：")
    for i, doc in enumerate(docs[:limit], start=1):
        print(f"    [{i}] chunk#{doc.metadata.get('chunk_id')}  {doc.page_content[:52]}...")


if dense_db is not None:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        # 权重写成 [0, 1] / [1, 0]，就把其中一路近似「关掉」，用来单独观察两路
        sparse_only = hybrid_db.as_retriever(search_kwargs={
            "k": 3, "fetch_k": 20,
            "ranker_type": "weighted",
            "ranker_params": {"weights": [0.0, 1.0]},
        })
        hybrid_even = hybrid_db.as_retriever(search_kwargs={
            "k": 3, "fetch_k": 20,
            "ranker_type": "weighted",
            "ranker_params": {"weights": [0.5, 0.5]},
        })

    for q in QUERIES:
        print(f"\n问题：{q}")
        show("稠密单路", dense_db.similarity_search(q, k=3))
        show("纯稀疏 weighted(0, 1)", sparse_only.invoke(q))
        show("等权 weighted(0.5, 0.5)", hybrid_even.invoke(q))
else:
    print("跳过：需要先完成上一格的建库")



问题：3099米海拔的是哪座峰
  稠密单路：
    [1] chunk#117  古有神鹿，步步莲花，今有云窝，处处不雪，真是妙不可言。

（十八）华严顶：不忘初心，方得始终

华严顶，...
    [2] chunk#131  九老洞是一个全封闭型的观赏空间，总共分为3个部分，浅部、中部和深部。现在我们走的这段是浅部，大家可以看到...
    [3] chunk#171  （三十一）千佛顶万佛顶：大千世界，万佛朝宗

万佛顶，海拔3099米，巍峨屹立在“大光明山”之巅。

现...


  纯稀疏 weighted(0, 1)：
    [1] chunk#170  卧云禅院是峨眉山最高峰仅存的古建筑寺院。明嘉靖年间性天和尚开建，因岩下白云环绕，寺卧云中，故名“卧云庵”...
    [2] chunk#1  峨眉山导游词

峨眉山概况

峨眉山，雄踞四川盆地西南边缘，素有“仙山佛国”、“植物王国”、“动物乐园”...
    [3] chunk#171  （三十一）千佛顶万佛顶：大千世界，万佛朝宗

万佛顶，海拔3099米，巍峨屹立在“大光明山”之巅。

现...
  等权 weighted(0.5, 0.5)：
    [1] chunk#171  （三十一）千佛顶万佛顶：大千世界，万佛朝宗

万佛顶，海拔3099米，巍峨屹立在“大光明山”之巅。

现...
    [2] chunk#170  卧云禅院是峨眉山最高峰仅存的古建筑寺院。明嘉靖年间性天和尚开建，因岩下白云环绕，寺卧云中，故名“卧云庵”...
    [3] chunk#38  四、峨眉山景点解说

（一）报国寺：庄严国土，报国主恩

报国寺，海拔533米，位于峨眉山麓下。这里是入...

问题：峨眉山哪座山峰最高


  稠密单路：
    [1] chunk#1  峨眉山导游词

峨眉山概况

峨眉山，雄踞四川盆地西南边缘，素有“仙山佛国”、“植物王国”、“动物乐园”...
    [2] chunk#72  到了隋开皇年间，天台宗始祖智者（公元538－597年）住持中峰寺，后来东去荆州，将衣钵锡杖留在寺中，成就...
    [3] chunk#69  神水灵山，福寿同源，有道是：泉与楚通，万法归宗，造物有灵，灵泉不穷。

（六）中峰寺：峨眉武源，谁与争峰...
  纯稀疏 weighted(0, 1)：
    [1] chunk#170  卧云禅院是峨眉山最高峰仅存的古建筑寺院。明嘉靖年间性天和尚开建，因岩下白云环绕，寺卧云中，故名“卧云庵”...
    [2] chunk#181  象池夜月

象池夜月，谭钟岳如是描述：“普贤骑像杳何之，胜迹空留洗象池。一月映池池贮月，月池感应妙难思。...
    [3] chunk#71  继明果大师之后，中峰寺高僧辈出，龙象非凡。北朝东魏战功显赫的“镇南将军”林时茂，因受权贵迫害，递发为僧，...


  等权 weighted(0.5, 0.5)：
    [1] chunk#1  峨眉山导游词

峨眉山概况

峨眉山，雄踞四川盆地西南边缘，素有“仙山佛国”、“植物王国”、“动物乐园”...
    [2] chunk#170  卧云禅院是峨眉山最高峰仅存的古建筑寺院。明嘉靖年间性天和尚开建，因岩下白云环绕，寺卧云中，故名“卧云庵”...
    [3] chunk#161  四面十方普贤像可以称得上佛教造像史上的奇迹，完美地体现了“千座名山一座顶”的佛教修行理念，是峨眉山普贤道...


对照三组结果就能看出互补性：

- 数字型问题里，稠密检索容易被「海拔」「山峰」这类高频语义带偏（3077 米的金顶很常见），纯稀疏和混合把带 3099 的答案块顶上来；
- 口语化问题里，纯稀疏只在「最高」这个词上有信号，稠密检索能把「极巅」「之巅」一起召回。

一个小技巧：`weighted` 的权重写成 `[0, 1]` 或 `[1, 0]`，就把其中一路近似「关掉」了——不带新建集合就能单独观察稠密、稀疏两路的表现。

`fetch_k=20` 是每路的**预取数量**：先各取 20 条，再由 ranker 融合成最终 `k` 条。`fetch_k` 太小时，融合只能在小池子里挑；太大则融合开销和噪声都上升。常用 `fetch_k = 4k ~ 8k`。

`fetch_k=20` 是每路的**预取数量**：先各取 20 条，再由 ranker 融合成最终 `k` 条。`fetch_k` 太小时，融合只能在小池子里挑；太大则融合开销和噪声都上升。常用 `fetch_k = 4k ~ 8k`。

这里显式传了 `weights=[0.5, 0.5]`（等权）。原因：不传 `ranker_type` 时，`langchain-milvus` 会默认构造 `WeightedRanker(1.0, 1.0)`，但在 `langchain-milvus 0.4.0 + pymilvus 3.0.3` 上实测会报服务器端错误：

```
MilvusException: the length of weights param mismatch with ann search requests: invalid parameter[expected=2][actual=3]
```

显式写出权重即可绕过。等权 weighted 也**不做归一化**，两路分数不同量纲时并不公平，这正是下一节要解决的问题。


## 6. 融合排序：weighted 与 RRF（`test6.py`）

### weighted：先加权求和

$$
s(d)=w_d\cdot\tilde s_{\text{dense}}(d)+w_s\cdot\tilde s_{\text{sparse}}(d)
$$

课堂注释里点出了它的两个致命缺陷：

1. **量纲要先归一化**。余弦相似度落在 $[-1,1]$，BM25 是开放式正数（可能几十），直接加权等于让 BM25 单方面决定排序；
2. **缺失的一路按 0 计**。某文档只被 BM25 召回到时：

$$
s = 0.6\times 0 + 0.4\times \tilde s_{\text{BM25}}
$$

即使它在语义上非常相关，也直接损失了 60% 的权重，很可能排不进 top-k。

### RRF：只看名次，不看分数

$$
\operatorname{RRF}(d)=\sum_{r\in\{\text{dense},\,\text{sparse}\}}\frac{1}{k+\operatorname{rank}_r(d)},\qquad k=60
$$

| 排名 | $1/\text{rank}$ | $1/(60+\text{rank})$ |
|---|---|---|
| 1 | 1.000 | 0.0164 |
| 2 | 0.500 | 0.0161 |
| 3 | 0.333 | 0.0159 |
| 10 | 0.100 | 0.0143 |
| 60 | 0.017 | 0.0083 |

为什么要有 $k=60$ 这个平滑项？直接取倒数时，第 1 名和第 2 名的贡献是 1.0 对 0.5，头名几乎垄断；加常数后 1/61 与 1/62 只差约 1.6%，让 top 区间贡献接近、尾部平滑衰减——**名次仍然重要，但不至于一处领先就通吃**。$k$ 越大越平滑，60 是经验默认值。

RRF 免疫量纲：它是「名次」的函数，不需要归一化；某一路没召回到的文档只损失这一路的贡献，不会被另一路的高分彻底压死。


In [10]:
dense_ranking = ["A", "B", "C", "D"]     # 稠密检索的名次
sparse_ranking = ["C", "A", "D", "E"]    # BM25 的名次（E 只在 BM25 出现）


def weighted_fuse(rankings, weights):
    """weighted 的近似：把 1/rank 当作归一化后的分数，再按权重相加。"""
    scores = {}
    for ranking, weight in zip(rankings, weights):
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + weight * (1.0 / rank)
    return dict(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))


def rrf_fuse(rankings, k=60):
    """RRF：1/(k+rank)，k=0 就退化为纯倒数。"""
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return dict(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))


print("两路名次：", {"dense": dense_ranking, "sparse": sparse_ranking})
print("\nweighted 0.6/0.4：", {d: round(s, 4) for d, s in weighted_fuse([dense_ranking, sparse_ranking], [0.6, 0.4]).items()})
print("RRF k=0（纯倒数） ：", {d: round(s, 4) for d, s in rrf_fuse([dense_ranking, sparse_ranking], k=0).items()})
print("RRF k=60          ：", {d: round(s, 4) for d, s in rrf_fuse([dense_ranking, sparse_ranking], k=60).items()})

print("\n注意 E：只在 BM25 出现")
print("  weighted 下 E = 0.4 × 1/4 =", round(0.4 * 0.25, 4), "，被一路缺失扣掉 60% 权重")
print("  RRF 下 E = 1/64 =", round(1 / 64, 4), "，只拿一路贡献，仍参与竞争")


两路名次： {'dense': ['A', 'B', 'C', 'D'], 'sparse': ['C', 'A', 'D', 'E']}

weighted 0.6/0.4： {'A': 0.8, 'C': 0.6, 'B': 0.3, 'D': 0.2833, 'E': 0.1}
RRF k=0（纯倒数） ： {'A': 1.5, 'C': 1.3333, 'D': 0.5833, 'B': 0.5, 'E': 0.25}
RRF k=60          ： {'A': 0.0325, 'C': 0.0323, 'D': 0.0315, 'B': 0.0161, 'E': 0.0156}

注意 E：只在 BM25 出现
  weighted 下 E = 0.4 × 1/4 = 0.1 ，被一路缺失扣掉 60% 权重
  RRF 下 E = 1/64 = 0.0156 ，只拿一路贡献，仍参与竞争


In [11]:
if hybrid_db is not None:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        weighted_retriever = hybrid_db.as_retriever(search_kwargs={
            "k": 4, "fetch_k": 20,
            "ranker_type": "weighted",
            "ranker_params": {"weights": [0.6, 0.4]},   # dense=0.6, sparse=0.4
        })
        rrf_retriever = hybrid_db.as_retriever(search_kwargs={
            "k": 4, "fetch_k": 20,
            "ranker_type": "rrf",
            "ranker_params": {"k": 60},
        })

    question = "金殿是哪一年建成的"
    print(f"问题：{question}")
    show("weighted(0.6, 0.4)", weighted_retriever.invoke(question), limit=4)
    show("RRF(k=60)", rrf_retriever.invoke(question), limit=4)
else:
    print("跳过：需要先完成第 5 节的建库")


问题：金殿是哪一年建成的
  weighted(0.6, 0.4)：
    [1] chunk#162  烧毁后的铁瓦殿是如何变身金殿的？这就不得不提妙峰禅师了。妙峰禅师与峨眉山的渊源肇始，来自于他的一个誓愿。...
    [2] chunk#164  到了清光绪十六年（公元1890年），一场大火，让金殿毁于一炬。光绪18年（公元1892年）寺僧心启、月照...
    [3] chunk#169  再往上走，便是峨眉山最高的殿堂——金殿。殿门的匾额书写着“金顶”二字，因整个金殿的檐、瓦等构件皆为铜质渗...
    [4] chunk#163  。因铜殿通体皆铜，屋顶檐瓦馏金，在阳光映照之下，金光闪闪，十分壮观，故称“金殿”或曰“金顶”。明神宗朱翊...


  RRF(k=60)：
    [1] chunk#163  。因铜殿通体皆铜，屋顶檐瓦馏金，在阳光映照之下，金光闪闪，十分壮观，故称“金殿”或曰“金顶”。明神宗朱翊...
    [2] chunk#162  烧毁后的铁瓦殿是如何变身金殿的？这就不得不提妙峰禅师了。妙峰禅师与峨眉山的渊源肇始，来自于他的一个誓愿。...
    [3] chunk#169  再往上走，便是峨眉山最高的殿堂——金殿。殿门的匾额书写着“金顶”二字，因整个金殿的檐、瓦等构件皆为铜质渗...
    [4] chunk#164  到了清光绪十六年（公元1890年），一场大火，让金殿毁于一炬。光绪18年（公元1892年）寺僧心启、月照...


### 版本坑：新旧两套 ranker 参数

`langchain-milvus 0.4.0` 把 `ranker_type` / `ranker_params` 标记为 **deprecated**，推荐改用 pymilvus 的 Function 对象：

```python
from pymilvus import RRFRanker
db.as_retriever(search_kwargs={"reranker": RRFRanker(60)})
```

但在 `langchain-milvus 0.4.0 + pymilvus 3.0.3` 实测会抛：

```
AttributeError: 'RRFRanker' object has no attribute 'type'
```

原因是 `langchain_milvus` 会检查 `reranker.type == FunctionType.RERANK`，而当前 pymilvus 的 `RRFRanker` 还没有这个属性。所以本讲仍然用旧参数（功能正常，只是每次会打 `DeprecationWarning`），等两边版本对齐后再迁移。

选型建议：默认用 RRF——不用调参、不怕量纲；只有当你确有先验（例如中文法律术语场景，稀疏更可信）时，才用 weighted，并且一定先做分数归一化。


## 7. 精排 Rerank：把 top-20 交给 Cross-Encoder（`test6.py` 后半）

| | 双塔 Bi-Encoder（召回用） | Cross-Encoder（精排用） |
|---|---|---|
| 输入 | query、doc **分别**编码 | $(q,d)$ 拼接后一起送入模型 |
| 交互 | 只在最后算一次相似度 | 每个词都能和对方每个词交互 |
| 能否预计算 | 可以（建库时算好文档向量） | 不能，每对都要现算 |
| 复杂度 | 近似检索近似 $O(1)$ | $O(N)$ 次前向，$N$ 为候选数 |
| 在链路中的位置 | 海选 top-50~100 | 复赛：top-10~20 精排后取 top-3 |

```mermaid
flowchart LR
    Q[query] --> R[召回 top-20<br/>混合 + RRF]
    R --> C[Cross-Encoder<br/>逐对打分]
    C --> T[取 top-3 进 Prompt]
    R -.->|召回漏掉的文档<br/>精排救不回来| X[无法补救]
```

课堂用本地模型 `rerank_qwen` 做精排：

```python
model = HuggingFaceCrossEncoder(model_name="/teacher_train/wtb/hug_model/rerank_qwen/")
compressor = CrossEncoderReranker(model=model, top_n=3)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever)
```

本机没有这个模型目录，所以下一格先保留课堂写法（默认不执行），再给一个云端等价实现。


In [12]:
RUN_LOCAL_RERANK = False   # 需要 sentence-transformers 和本地 rerank_qwen 模型目录

if RUN_LOCAL_RERANK:
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

    # 课堂路径：/teacher_train/wtb/hug_model/rerank_qwen/
    cross_encoder = HuggingFaceCrossEncoder(model_name="./rerank_qwen")
    local_compressor = CrossEncoderReranker(model=cross_encoder, top_n=3)
    local_compression_retriever = ContextualCompressionRetriever(
        base_compressor=local_compressor,
        base_retriever=rrf_retriever,
    )
    print(local_compression_retriever.invoke("峨眉山海拔最高的山峰"))
else:
    print("跳过本地精排：本机无 rerank_qwen 模型；下一格用 DashScope gte-rerank-v2 实现同样的接口")


跳过本地精排：本机无 rerank_qwen 模型；下一格用 DashScope gte-rerank-v2 实现同样的接口


In [13]:
import requests
from langchain_core.documents.compressor import BaseDocumentCompressor


class DashScopeRerankCompressor(BaseDocumentCompressor):
    """用 DashScope 的 gte-rerank 重新打分：接口形状与 CrossEncoderReranker 一致。"""

    model: str = "gte-rerank-v2"
    top_n: int = 3
    api_key: str = ""
    endpoint: str = "https://dashscope.aliyuncs.com/api/v1/services/rerank/text-rerank/text-rerank"
    timeout: int = 30

    def compress_documents(self, documents, query, callbacks=None):
        if not documents:
            return []
        response = requests.post(
            self.endpoint,
            headers={"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"},
            json={
                "model": self.model,
                "input": {"query": query, "documents": [d.page_content for d in documents]},
                "parameters": {"return_documents": False, "top_n": min(self.top_n, len(documents))},
            },
            timeout=self.timeout,
        )
        response.raise_for_status()
        results = response.json()["output"]["results"]      # 已按 relevance_score 降序
        ranked = []
        for item in results:
            doc = documents[item["index"]].model_copy(deep=True)
            doc.metadata["rerank_score"] = round(item["relevance_score"], 4)
            ranked.append(doc)
        return ranked


compression_retriever = None
if hybrid_db is not None:
    from langchain_classic.retrievers import ContextualCompressionRetriever

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        recall_retriever = hybrid_db.as_retriever(search_kwargs={
            "k": 6, "fetch_k": 20, "ranker_type": "rrf", "ranker_params": {"k": 60},
        })
    reranker = DashScopeRerankCompressor(api_key=lab_config.DASHSCOPE_API_KEY, top_n=3)
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=reranker,
        base_retriever=recall_retriever,
    )

    question = "金顶的佛光是怎么形成的"
    before = recall_retriever.invoke(question)
    after = compression_retriever.invoke(question)
    show("精排前（混合召回 k=6）", before, limit=6)
    print("  精排后（top_n=3，附 relevance_score）：")
    for i, doc in enumerate(after, start=1):
        print(f"    [{i}] {doc.metadata['rerank_score']:.4f}  chunk#{doc.metadata.get('chunk_id')}  "
              f"{doc.page_content[:48]}...")
else:
    print("跳过：需要先完成第 5 节的建库")


  精排前（混合召回 k=6）：
    [1] chunk#186  传说虽然虚幻遥远，但这一奇景还是确实存在的。适逢天气晴朗，在金顶摄身崖附近，当人站在金顶背向太阳而立，太...
    [2] chunk#187  就佛家而言，将这自然奇观称之为“佛光”，给予了一个美好的称谓，在峨眉山一直流传下来，这也是信仰的需要。其...
    [3] chunk#185  在金顶祥光中，最神秘的非“佛光”莫属了。而佛光与蒲公逐鹿登顶的传奇见闻还有点关联。相传，峨眉山华严顶下住...
    [4] chunk#37  现在我们前往的是接引殿。接引殿，供奉着接引佛，意在西方净土，接引彼岸。接引佛，又称阿弥陀佛。根据佛教教义...
    [5] chunk#182  灵岩叠翠

灵岩地处峨眉金顶三峰的后山麓，隋唐年间在这里创建了灵岩寺。灵岩寺传为印度僧人宝掌结茅处，鼎盛...
    [6] chunk#42  走过这山门，迎面第一殿便是弥勒殿，匾额上的“宝相庄严”四字为原中国佛教协会会长赵朴初居士所书。殿堂内供奉...
  精排后（top_n=3，附 relevance_score）：
    [1] 0.5481  chunk#186  传说虽然虚幻遥远，但这一奇景还是确实存在的。适逢天气晴朗，在金顶摄身崖附近，当人站在金顶背向太阳...
    [2] 0.4565  chunk#187  就佛家而言，将这自然奇观称之为“佛光”，给予了一个美好的称谓，在峨眉山一直流传下来，这也是信仰的...
    [3] 0.3475  chunk#185  在金顶祥光中，最神秘的非“佛光”莫属了。而佛光与蒲公逐鹿登顶的传奇见闻还有点关联。相传，峨眉山华...


### 读精排结果的三个要点

1. **`relevance_score` 是模型打分，不是概率**：0.94 与 0.41 说明「更像答案」，但不能解读成 94% 正确率；
2. **精排只改顺序，不增加召回**：图上那条虚线是重点——召回阶段没进来的文档，精排无从补救。所以 `fetch_k`、`k` 要给足，再让精排做减法；
3. **成本换质量**：每多一个候选就多一次前向（或一次 API token）。常见组合是召回 20、精排 3~5，而不是召回 100。

课堂的 `CrossEncoderReranker(top_n=3)` 与本格 `DashScopeRerankCompressor(top_n=3)` 在链里的角色完全相同——都是 `ContextualCompressionRetriever` 的 `base_compressor`，只是底层模型一本地一云端。


## 8. 查询改写：一次提问，多路召回（`test7.py`）

`MultiQueryRetriever` 的思路是「问法不好，先换个问法」：让 LLM 把原问题改写成若干个不同角度的版本，各自检索，最后去重合并。

```mermaid
flowchart LR
    Q[原始问题] --> L[LLM 改写]
    L --> Q1[改写 1] & Q2[改写 2] & Q3[改写 3] & Q4[改写 4]
    Q1 --> R[检索] & Q2 --> R & Q3 --> R & Q4 --> R
    R --> D[去重合并]
```

课堂的自定义 prompt（`test7.py` 原样）：

```python
CUSTOM_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""你是一个专业的问题改写助手。请为下面的问题生成4个不同的改写版本，
每个版本应该：
- 从不同角度表达相同的意图
- 使用不同的关键词和表达方式
- 保持问题的核心含义

每个问题单独一行，不要编号。

原始问题: {question}

改写后的问题:"""
)
```

| 项目 | 代价 |
|---|---|
| 改写本身 | 1 次 LLM 调用 |
| 检索 | N 次（N = 改写条数，通常 3~5） |
| 收益 | 召回同义表达 / 不同切面，缓解「问法不专业」 |
| 风险 | 改写跑偏会把噪声带进来，最好配合精排一起用 |

课上默认 prompt 是英文的（`from_llm` 默认参数），中文语料建议像课堂那样传自定义 prompt；`include_original=True` 可以让原问题也参与检索。


In [14]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.prompts import PromptTemplate

CUSTOM_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""你是一个专业的问题改写助手。请为下面的问题生成4个不同的改写版本，
每个版本应该：
- 从不同角度表达相同的意图
- 使用不同的关键词和表达方式
- 保持问题的核心含义

每个问题单独一行，不要编号。

原始问题: {question}

改写后的问题:""",
)

if hybrid_db is not None:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        base_retriever = hybrid_db.as_retriever(search_kwargs={
            "k": 5, "fetch_k": 20, "ranker_type": "rrf", "ranker_params": {"k": 60},
        })

    question = "金顶的佛光是怎么形成的"
    print("== 模型生成的改写 ==")
    print(llm.invoke(CUSTOM_PROMPT.format(question=question)).content.strip())

    multi_retriever = MultiQueryRetriever.from_llm(
        retriever=base_retriever,
        llm=llm,
        prompt=CUSTOM_PROMPT,
    )
    single_docs = base_retriever.invoke(question)
    multi_docs = multi_retriever.invoke(question)

    single_ids = {d.metadata["chunk_id"] for d in single_docs}
    multi_ids = {d.metadata["chunk_id"] for d in multi_docs}
    print(f"\n原问题检索：{len(single_docs)} 条 / chunk {sorted(single_ids)}")
    print(f"改写后合并：{len(multi_docs)} 条 / chunk {sorted(multi_ids)}")
    print(f"新增召回：{sorted(multi_ids - single_ids)}")
else:
    print("跳过：需要先完成第 5 节的建库")


== 模型生成的改写 ==


峨眉山金顶出现佛光现象的科学原理是什么
请解释一下金顶佛光这一自然奇观的成因机制
从光学角度分析，金顶上的佛光是如何产生的
揭秘峨眉山金顶佛光形成的具体过程



原问题检索：5 条 / chunk [37, 182, 185, 186, 187]
改写后合并：10 条 / chunk [37, 115, 161, 162, 182, 183, 184, 185, 186, 187]
新增召回：[115, 161, 162, 183, 184]


改写后召回变多是常态，但**多不等于好**：新增的块可能只是「词面相关」。工程上通常把 MultiQuery 放在召回阶段，紧接着用 RRF 融合、再用精排收敛，链路就变成：

```
问题 → LLM 改写（N 条）→ 每条各做混合检索 → RRF 融合 → Cross-Encoder top_n
```

这也是第 6、7 节先讲融合与精排、再讲改写的原因：改写放大召回，融合和精排负责把噪声收回来。


## 9. 检索器转工具：把「查资料」交给 Agent（`test2.py`）

`create_retriever_tool` 把检索器包装成一个标准工具，交给 Agent 决定何时调用：

```python
retriever_tool = create_retriever_tool(
    res,                                  # 任意 BaseRetriever
    name="car_info_retriever",            # 工具标识符：字母数字下划线，给代码和日志用
    description="查询领克汽车的信息",       # 给 LLM 看的「什么时候该调用我」
)
```

课堂里 `name` 与 `description` 都在描述「领克汽车」，而底层 collection 是 `car_info`——这说明三者是独立的：

| 字段 | 作用对象 | 写错的后果 |
|---|---|---|
| `name` | 代码 / 工具调用协议 | 重名冲突、调用报错 |
| `description` | LLM 的路由判断 | 该查资料时不查，或不该查时乱查 |
| 底层 collection | 实际数据 | 与描述不一致 → 答非所问，且难排查 |

经验：`description` 里写清**资料范围 + 典型问题**（例如「峨眉山景区讲解：寺院、海拔、景观、传说」），比只写「查询信息」有效得多。


In [15]:
from langchain_core.tools import create_retriever_tool

if compression_retriever is not None:
    emei_tool = create_retriever_tool(
        compression_retriever,
        name="emei_guide_retriever",
        description="查询峨眉山景区讲解资料：寺院历史、海拔、十景、佛光与传说。输入应是具体问题。",
    )
    print("工具名：", emei_tool.name)
    print("工具描述：", emei_tool.description)
    output = emei_tool.invoke({"query": "金顶祥光包括哪几种奇观"})
    print("\n== 工具返回（节选）==")
    print(output[:400])
else:
    print("跳过：需要先完成第 7 节的压缩检索器")


工具名： emei_guide_retriever
工具描述： 查询峨眉山景区讲解资料：寺院历史、海拔、十景、佛光与传说。输入应是具体问题。



== 工具返回（节选）==
在金顶祥光中，最神秘的非“佛光”莫属了。而佛光与蒲公逐鹿登顶的传奇见闻还有点关联。相传，峨眉山华严顶下住着一位采药老人，名叫蒲公。一日，蒲公正在林间采药，突然发现有鹿的脚印，而且这鹿的脚印似莲花般绚烂，便跟随着鹿迹，爬到了山顶。到了山顶，鹿迹消失了，抬头一看，舍身崖下云海翻卷，彩光万道。眼前白色的云层上，还平卧着一个耀人的绚丽光环，有着赤、橙、黄、绿、青、蓝、紫七种颜色。在这光环的中央，竟然还站着一个黄暗虚明的人影。这人头戴束发紫金冠，身披黄锦袈裟，骑着一匹六牙白象，脚下是白玉莲台。后来得知，这是普贤菩萨现的金身，显了瑞像。

灵岩叠翠

灵岩地处峨眉金顶三峰的后山麓，隋唐年间在这里创建了灵岩寺。灵岩寺传为印度僧人宝掌结茅处，鼎盛时期曾有“九处过堂，十处烧香，烧香要骑马”的说法。到了明代，灵岩寺尤为鼎盛，光殿宇就有四十八重，密林掩映，丹岩凝翠，呈现出层层叠叠的雄峙壮景，古称“灵岩叠翠”。但


如果把它接进 Agent，模型看到的就是「工具名 + 描述 + 参数」，由它判断要不要先查资料再回答。这也是 1109 讲里「检索器转工具」的同一套路，只是这一讲的检索器外面多了融合与精排两层。

自己调用时 `invoke({"query": ...})` 返回的是**拼好的字符串**（按 `document_separator` 连接块内容），不是 `Document` 列表——需要元数据时直接调底层 retriever。


## 10. RAGAS 评估：把「答得好不好」拆成两个指标（`ragas_test.py`）

```mermaid
flowchart LR
    Q[问题] --> R[检索 contexts] --> A[生成 answer]
    Q --> F
    A --> F[Faithfulness<br/>把 answer 拆成若干 claim<br/>逐条问：能否由 contexts 推出？]
    Q --> C
    C[ContextRecall<br/>把 ground_truth 拆成若干 claim<br/>逐条问：contexts 覆盖到了吗？] --> C
```

| 指标 | 输入 | 衡量什么 | 低了说明 |
|---|---|---|---|
| **Faithfulness**（忠实度） | 问题、答案、contexts | 答案里的每条结论是否都能由检索到的原文支持 | 模型在编 / 混入外部知识 / 上下文有噪声 |
| **ContextRecall**（上下文召回率） | 问题、标准答案、contexts | 标准答案里的信息点有多少被 contexts 覆盖 | **检索没捞全**——该换检索策略（混合、改写、加大 k） |

这两个指标的分工正好对应 RAG 的两段：召回好、忠实度低 → 问题在生成；召回低 → 问题在检索。诊断顺序建议先看 ContextRecall，再看 Faithfulness。

RAGAS 0.4 的调用形状（课堂原版用 Ollama 当裁判，这里换云端千问）：

```python
judge = llm_factory(model, client=AsyncOpenAI(base_url=..., api_key=...), max_tokens=8192)
faithfulness = Faithfulness(llm=judge)
context_recall = ContextRecall(llm=judge)

# 普通 .py 脚本（无运行中的事件循环）：
faithfulness.score(user_input=q, response=answer, retrieved_contexts=contexts).value
context_recall.score(user_input=q, reference=ground_truth, retrieved_contexts=contexts).value

# Notebook 单元格里已经有一个运行中的事件循环，同步版会报
# RuntimeError: Cannot call sync score() from an async context. Use ascore() instead.
await faithfulness.ascore(user_input=q, response=answer, retrieved_contexts=contexts)
```

**兼容性提醒**：`ragas 0.4.3` 在 `ragas/llms/base.py` 里仍然 `from langchain_community.chat_models.vertexai import ChatVertexAI`，而 `langchain-community 0.4.x` 已经移除了这个模块。下一格用一段 shim 顶上去（`ChatVertexAI` 只出现在 `isinstance` 列表里，不影响指标计算）。


In [16]:
if RUN_EVAL:
    import types

    if "langchain_community.chat_models.vertexai" not in sys.modules:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", DeprecationWarning)
                import langchain_community.chat_models.vertexai  # noqa: F401
        except ModuleNotFoundError:
            shim = types.ModuleType("langchain_community.chat_models.vertexai")

            class ChatVertexAI:   # 占位类：只参与 isinstance 判断
                ...

            shim.ChatVertexAI = ChatVertexAI
            sys.modules["langchain_community.chat_models.vertexai"] = shim

    from openai import AsyncOpenAI
    from ragas.llms import llm_factory
    from ragas.metrics.collections import ContextRecall, Faithfulness

    judge = llm_factory(
        lab_config.MODEL,
        client=AsyncOpenAI(base_url=lab_config.BASE_URL, api_key=lab_config.API_KEY),
        max_tokens=4096,
        extra_body={"enable_thinking": False},   # 千问思考模式会吃掉 token 预算
    )
    faithfulness = Faithfulness(llm=judge)
    context_recall = ContextRecall(llm=judge)
    print(f"RAGAS 裁判：{lab_config.MODEL}")
else:
    print("跳过：RUN_EVAL=False")


RAGAS 裁判：qwen3.8-flash


In [17]:
EVAL_SET = [
    {"question": "峨眉山最高峰是哪座，海拔多少米？",
     "ground_truth": "峨眉山最高峰是万佛顶，海拔3099米。"},
    {"question": "四面十方普贤像通高多少米，总重多少吨？",
     "ground_truth": "四面十方普贤像通高48米，总重660吨。"},
    {"question": "金顶祥光由哪四大自然奇观组成？",
     "ground_truth": "金顶祥光由日出、云海、佛光、圣灯四大自然奇观组成。"},
]

ANSWER_PROMPT = PromptTemplate(
    input_variables=["question", "context_text"],
    template="""基于以下上下文信息回答问题。如果上下文中没有相关信息，请说"无法从提供的信息中回答"。

上下文信息：
{context_text}

问题：{question}

答案：""",
)

eval_rows = []
if RUN_EVAL and compression_retriever is not None:
    for item in EVAL_SET:
        contexts = [doc.page_content for doc in compression_retriever.invoke(item["question"])]
        answer = llm.invoke(ANSWER_PROMPT.format(
            question=item["question"],
            context_text="\n\n".join(contexts),
        )).content.strip()

        faith_score = (await faithfulness.ascore(
            user_input=item["question"], response=answer, retrieved_contexts=contexts,
        )).value
        recall_score = (await context_recall.ascore(
            user_input=item["question"], reference=item["ground_truth"], retrieved_contexts=contexts,
        )).value

        eval_rows.append({"question": item["question"], "answer": answer,
                          "faithfulness": faith_score, "context_recall": recall_score})
        print(f"\n问题：{item['question']}")
        print(f"答案：{answer}")
        print(f"faithfulness={faith_score:.3f}  context_recall={recall_score:.3f}")

    if eval_rows:
        mean_f = sum(r["faithfulness"] for r in eval_rows) / len(eval_rows)
        mean_r = sum(r["context_recall"] for r in eval_rows) / len(eval_rows)
        print(f"\n均值：faithfulness={mean_f:.3f}  context_recall={mean_r:.3f}")

    # 对照实验：把一条「编造数字」的答案也打一次分，验证指标能抓住幻觉
    if eval_rows:
        bad_contexts = [doc.page_content for doc in compression_retriever.invoke(EVAL_SET[0]["question"])]
        bad_answer = "峨眉山最高峰是万佛顶，海拔4099米。"
        bad_score = (await faithfulness.ascore(
            user_input=EVAL_SET[0]["question"], response=bad_answer, retrieved_contexts=bad_contexts,
        )).value
        print(f"\n对照（编造海拔）：{bad_answer}")
        print(f"faithfulness={bad_score:.3f}，同一问题正常答案是 {eval_rows[0]['faithfulness']:.3f}")
else:
    print("跳过：需要 RUN_EVAL=True 且先完成第 7 节的检索器")



问题：峨眉山最高峰是哪座，海拔多少米？
答案：峨眉山最高峰是万佛顶，海拔3099米。
faithfulness=1.000  context_recall=1.000



问题：四面十方普贤像通高多少米，总重多少吨？
答案：四面十方普贤像通高48米，总重660吨。
faithfulness=1.000  context_recall=1.000



问题：金顶祥光由哪四大自然奇观组成？
答案：根据上下文信息，金顶祥光由**日出、云海、佛光、圣灯**这四大自然奇观组成。
faithfulness=1.000  context_recall=1.000

均值：faithfulness=1.000  context_recall=1.000



对照（编造海拔）：峨眉山最高峰是万佛顶，海拔4099米。
faithfulness=0.500，同一问题正常答案是 1.000


### 读分数与几个坑

- **先看 ContextRecall**：它低说明信息没进上下文，后面 Faithfulness 再高也没意义（模型在「只用手里的材料」上做得再好，材料本身就不全）。
- **对照样本验证了指标在工作**：把正常答案换成编造的「海拔 4099 米」，faithfulness 立刻从 1.0 掉下来——说明它不是永远给满分。
- **Faithfulness 高不代表答对**：只要答案没超出 contexts，哪怕 contexts 里没有正确答案，它也可能得高分——它衡量「不瞎编」，不衡量「正确」。
- **裁判本身有噪声**：换裁判模型、换提示词，分数都会动。0.62 和 0.68 的差别不要当结论，看的是同一套配置下的**趋势**。
- **中文支持**：RAGAS 默认提示词是英文的，中文语料上的判定会有波动，样本量小时建议 5~10 条题 + 人工抽查 2~3 条。
- **成本**：每个样本 2 次裁判调用（每次内部还要多次拆 claim），所以 `RUN_EVAL` 单独开关。

这三条题刚好覆盖三种情况：数字题（BM25 有优势）、专名 + 数字题、列举题（需要一整段连续文本，考验切块粒度）。


## 11. 收尾：一张调参清单

| 症状 | 先动哪一层 | 具体动作 | 对应课堂文件 |
|---|---|---|---|
| ContextRecall 低 | 召回 | `fetch_k` 调到 30~50；确认 BM25 已开；加 MultiQuery | `test5.py` `test7.py` |
| 数字 / 型号 / 专名答错 | 召回 | 维护 jieba 自定义词典；确认 sparse 字段已建；提高 sparse 权重 | `test4.py` `test5.py` |
| 召回够但仍答错 | 生成 | Prompt 要求「只用资料 + 引用编号」；降低温度；换更大模型 | `ragas_test.py` 里的 `ANSWER_PROMPT` |
| Faithfulness 低 | 上下文 | 收窄上下文（精排 `top_n=3`）；去掉低分块；要求「资料不足就说不知道」 | `test6.py` |
| 延迟 / 成本高 | 在线链路 | 去掉 MultiQuery（省 1 次 LLM）；精排只排 top-10；`fetch_k` 回到 20 | `test6.py` `test7.py` |
| 换库 / 迁移 | 存储 | Chroma → Milvus 时注意一致性级别与 flush 行为 | `test1.py` `test3.py` |

| 组件 | 额外延迟 | 额外成本 |
|---|---|---|
| 稀疏 BM25 | 毫秒级（服务端） | 存储一份 sparse 向量 |
| RRF 融合 | 可忽略 | 无 |
| Cross-Encoder 精排 | $N$ 次前向（$N$ = 候选数） | 云端按 token 计费 |
| MultiQuery | +1 次 LLM + $N$ 次检索 | 明显 |
| RAGAS 评测 | 离线 | 每样本 2 次裁判调用 |

一句话总结这一讲：**召回负责「不遗漏」，融合负责「公平比较」，精排负责「精确排序」，改写负责「换问法」，评估负责「知道哪一层坏了」**。


In [18]:
# 需要清理课堂集合时再取消注释（默认保留，方便下一讲继续实验）
# from pymilvus import MilvusClient
#
# client = MilvusClient(uri=MILVUS_URI)
# for name in ("emei_demo", DENSE_COLLECTION, HYBRID_COLLECTION):
#     if client.has_collection(name):
#         client.drop_collection(name)
# print(client.list_collections())

print("本讲结束：集合 emei_demo / emei_dense / emei_hybrid 保留在本地 Milvus，可继续做实验。")


本讲结束：集合 emei_demo / emei_dense / emei_hybrid 保留在本地 Milvus，可继续做实验。
